In [1]:
import os
import glob
import pandas as pd
import numpy as np
from scipy import stats
from statsmodels.stats.multicomp import pairwise_tukeyhsd

# =========================================================
# CONFIG
# =========================================================
root = os.path.abspath('../..')

rawdy_path = f'{root}/data/rawdy/rawdy_savitzky_lrs70'
bp_file = f'{root}/data/dgh_fwl_estimation_48000_savitzky.csv'   # ajusta la ruta si hace falta

vp_col = 'Vertical Position m'
ec_col = 'Corrected sp Cond [µS/cm]'
id_col = 'ID'
bp2_col = 'breakpoint_2 (vp)'

alpha = 0.05
max_shapiro_n = 500  # muestreo para evitar que Shapiro sea demasiado sensible por n enorme


# =========================================================
# 1. LEER BREAKPOINTS
# =========================================================
df_bp = pd.read_csv(bp_file)
df_bp = df_bp.dropna(subset=[id_col, bp2_col]).copy()


# =========================================================
# 2. ENCONTRAR ARCHIVO CRUDO POR ID
# =========================================================
def find_profile_file(profile_id, folder):
    pattern = os.path.join(folder, f'{profile_id}*.csv')
    matches = glob.glob(pattern)

    if len(matches) == 0:
        return None
    if len(matches) > 1:
        print(f'Advertencia: múltiples archivos para {profile_id}. Se usará el primero:')
        for m in matches:
            print(f'  - {m}')
    return matches[0]


# =========================================================
# 3. EXTRAER DATOS CRUDOS DE SZ SEGÚN bp2_vp
# =========================================================
all_sz_data = []

for _, row in df_bp.iterrows():
    profile_id = row[id_col]
    bp2_vp = row[bp2_col]

    profile_file = find_profile_file(profile_id, rawdy_path)
    if profile_file is None:
        print(f'No se encontró archivo para ID: {profile_id}')
        continue

    df_profile = pd.read_csv(profile_file)

    missing_cols = [c for c in [vp_col, ec_col] if c not in df_profile.columns]
    if missing_cols:
        print(f'Archivo {profile_file} no tiene columnas requeridas: {missing_cols}')
        continue

    # SZ definida como vp > breakpoint_2
    sz_mask = df_profile[vp_col] > bp2_vp
    df_sz = df_profile.loc[sz_mask, [vp_col, ec_col]].copy()
    df_sz = df_sz.dropna(subset=[vp_col, ec_col])

    if df_sz.empty:
        print(f'La SZ quedó vacía para ID: {profile_id}')
        continue

    df_sz['ID'] = profile_id
    df_sz['date'] = df_sz['ID'].str.extract(r'(\d{8})')

    all_sz_data.append(df_sz)

if not all_sz_data:
    raise ValueError('No se pudo construir ningún dataset de SZ.')

df_sz_all = pd.concat(all_sz_data, ignore_index=True)

# quitar filas sin fecha por seguridad
df_sz_all = df_sz_all.dropna(subset=['date']).copy()

print('\n=== CONTEOS POR ID ===')
print(df_sz_all.groupby('ID')[ec_col].count())

print('\n=== CONTEOS POR FECHA ===')
print(df_sz_all.groupby('date')[ec_col].count())


# =========================================================
# 4. RESUMEN DESCRIPTIVO
# =========================================================
summary = (
    df_sz_all.groupby('date')[ec_col]
    .agg(['count', 'mean', 'std', 'min', 'median', 'max'])
    .reset_index()
)

print('\n=== RESUMEN DESCRIPTIVO POR FECHA ===')
print(summary)


# =========================================================
# 5. PREPARAR GRUPOS
# =========================================================
dates = sorted(df_sz_all['date'].unique())
groups = [df_sz_all.loc[df_sz_all['date'] == d, ec_col].values for d in dates]

if len(groups) < 2:
    raise ValueError('Se necesitan al menos dos grupos con datos para comparar.')


# =========================================================
# 6. VALIDACIÓN DE SUPUESTOS
# =========================================================
print('\n=== NORMALIDAD (SHAPIRO-WILK) ===')
shapiro_results = []

for d in dates:
    sample = df_sz_all.loc[df_sz_all['date'] == d, ec_col]

    # Muestreo para evitar sensibilidad extrema por tamaños enormes
    if len(sample) > max_shapiro_n:
        sample = sample.sample(max_shapiro_n, random_state=0)

    stat, p = stats.shapiro(sample)
    shapiro_results.append({'date': d, 'stat': stat, 'p': p})

    conclusion = 'compatible con normalidad' if p >= alpha else 'desviación de normalidad'
    print(f'{d}: W = {stat:.6f}, p = {p:.6e} -> {conclusion}')

all_shapiro_ok = all(r['p'] >= alpha for r in shapiro_results)

print('\n=== HOMOGENEIDAD DE VARIANZAS (LEVENE) ===')
levene_stat, levene_p = stats.levene(*groups)
levene_ok = levene_p >= alpha

if levene_ok:
    print(f'Levene: Stat = {levene_stat:.6f}, p = {levene_p:.6e} -> varianzas homogéneas')
else:
    print(f'Levene: Stat = {levene_stat:.6f}, p = {levene_p:.6e} -> varianzas diferentes')


# =========================================================
# 7. SELECCIÓN AUTOMÁTICA DEL TEST GLOBAL
# =========================================================
print('\n=== SELECCIÓN AUTOMÁTICA DEL TEST GLOBAL ===')

# Reglas prácticas:
# - Si Levene pasa, usar ANOVA clásico
# - Si Levene no pasa, usar Welch ANOVA
# - Shapiro se reporta, pero no bloquea por n grande

test_used = None
global_p = None

if levene_ok:
    f_stat, p_anova = stats.f_oneway(*groups)
    test_used = 'ANOVA clásico'
    global_p = p_anova

    print(f'Se usará {test_used} porque Levene no detectó diferencias significativas en varianzas.')
    print(f'F = {f_stat:.6f}')
    print(f'p = {p_anova:.6e}')
else:
    # Welch ANOVA con pingouin
    try:
        import pingouin as pg

        welch = pg.welch_anova(
            dv=ec_col,
            between='date',
            data=df_sz_all
        )

        test_used = 'Welch ANOVA'
        global_p = welch['p_unc'].iloc[0]

        print(f'Se usará {test_used} porque Levene detectó heterogeneidad de varianzas.')
        print(welch)

    except ImportError:
        print('No está instalado pingouin. Instálalo con: pip install pingouin')
        raise


# =========================================================
# 8. CONCLUSIÓN AUTOMÁTICA DEL TEST GLOBAL
# =========================================================
print('\n=== CONCLUSIÓN DEL TEST GLOBAL ===')

if global_p < alpha:
    print(
        f'Como p = {global_p:.6e} < {alpha}, se rechaza la hipótesis nula de igualdad de medias. '
        'Existe evidencia de diferencias estadísticamente significativas en la conductividad de la SZ entre fechas.'
    )
else:
    print(
        f'Como p = {global_p:.6e} >= {alpha}, no se rechaza la hipótesis nula. '
        'No hay evidencia suficiente para afirmar diferencias estadísticamente significativas en la conductividad de la SZ entre fechas.'
    )


# =========================================================
# 9. POST-HOC AUTOMÁTICO
# =========================================================
print('\n=== POST-HOC ===')

if global_p < alpha:
    if test_used == 'ANOVA clásico':
        # Tukey es apropiado si usaste ANOVA clásico con varianzas homogéneas
        tukey = pairwise_tukeyhsd(
            endog=df_sz_all[ec_col],
            groups=df_sz_all['date'],
            alpha=alpha
        )

        print('Post-hoc seleccionado: Tukey HSD')
        print(tukey)

        tukey_df = pd.DataFrame(
            data=tukey._results_table.data[1:],
            columns=tukey._results_table.data[0]
        )

        signif_pairs = tukey_df[tukey_df['reject'] == True].copy()

        print('\n=== CONCLUSIÓN AUTOMÁTICA DEL POST-HOC ===')
        if signif_pairs.empty:
            print('Aunque el test global fue significativo, Tukey no encontró pares con diferencia significativa al corregir por comparaciones múltiples.')
        else:
            print('Tukey encontró diferencias significativas entre los siguientes pares de fechas:')
            for _, r in signif_pairs.iterrows():
                print(
                    f"- {r['group1']} vs {r['group2']}: "
                    f"diferencia de medias = {r['meandiff']:.3f}, p-ajustada = {r['p-adj']:.6e}"
                )

    elif test_used == 'Welch ANOVA':
        # Games-Howell es más adecuado cuando las varianzas no son homogéneas
        try:
            import pingouin as pg

            gh = pg.pairwise_gameshowell(
                data=df_sz_all,
                dv=ec_col,
                between='date'
            )

            print('Post-hoc seleccionado: Games-Howell')
            print(gh)

            signif_pairs = gh[gh['pval'] < alpha].copy()

            print('\n=== CONCLUSIÓN AUTOMÁTICA DEL POST-HOC ===')
            if signif_pairs.empty:
                print('Welch ANOVA fue significativo, pero Games-Howell no detectó pares con diferencia significativa.')
            else:
                print('Games-Howell encontró diferencias significativas entre los siguientes pares de fechas:')
                for _, r in signif_pairs.iterrows():
                    print(
                        f"- {r['A']} vs {r['B']}: "
                        f"diferencia de medias = {r['diff']:.3f}, p = {r['pval']:.6e}"
                    )

        except ImportError:
            print('No está instalado pingouin. Instálalo con: pip install pingouin')
            raise
else:
    print('No se ejecuta post-hoc porque el test global no fue significativo.')


# =========================================================
# 10. INTERPRETACIÓN GENERAL AUTOMÁTICA
# =========================================================
print('\n=== INTERPRETACIÓN GENERAL ===')

if not all_shapiro_ok:
    print(
        'Shapiro-Wilk detectó cierta desviación de normalidad en al menos uno de los grupos. '
        'Sin embargo, dado el tamaño muestral grande, esto no necesariamente invalida el análisis global.'
    )
else:
    print('Los grupos evaluados son compatibles con normalidad según Shapiro-Wilk.')

if levene_ok:
    print(
        'Las varianzas entre grupos pueden considerarse homogéneas; por ello, el uso de ANOVA clásico y Tukey HSD es apropiado.'
    )
else:
    print(
        'Las varianzas entre grupos no son homogéneas; por ello, el uso de Welch ANOVA y Games-Howell es más apropiado que ANOVA clásico con Tukey.'
    )

if global_p < alpha:
    print(
        'En conjunto, los resultados respaldan que la conductividad de la zona salina (SZ) presenta variabilidad temporal entre campañas.'
    )
else:
    print(
        'En conjunto, los resultados no muestran evidencia suficiente de variabilidad temporal estadísticamente significativa en la conductividad de la zona salina (SZ).'
    )


=== CONTEOS POR ID ===
ID
LRS70_D_YSI_20220131      1252
LRS70_D_YSI_20220816       320
LRS70_D_YSI_20230822       458
LRS70_D_YSI_20251123      1428
LRS70_D_YSI_R_20250226     262
Name: Corrected sp Cond [µS/cm], dtype: int64

=== CONTEOS POR FECHA ===
date
20220131    1252
20220816     320
20230822     458
20250226     262
20251123    1428
Name: Corrected sp Cond [µS/cm], dtype: int64

=== RESUMEN DESCRIPTIVO POR FECHA ===
       date  count          mean         std           min        median  \
0  20220131   1252  54474.297314  740.649883  52990.900951  54758.516931   
1  20220816    320  53934.946121  141.766601  53683.481102  53944.110086   
2  20230822    458  53920.756018  246.844683  53354.072186  53967.217295   
3  20250226    262  54255.282656  184.681158  53811.603637  54286.317752   
4  20251123   1428  54866.017479  449.118537  53801.722217  54987.586740   

            max  
0  55378.801344  
1  54232.353496  
2  54287.335280  
3  54515.976447  
4  55422.793526  

=== 

In [ ]:
# medias por fecha, en µS/cm
means_uS = {
    '20220131': 54195.907030,
    '20220816': 52375.391876,
    '20230822': 52724.482498,
    '20250226': 53459.408651,
    '20251123': 54050.810400,
}

# accuracy = ±0.5% de la lectura para 0–100 mS/cm (manual EXO, página 84)
# como los datos están en µS/cm, se puede aplicar igual:
# U = 0.005 * lectura
acc_uS = {k: 0.005 * v for k, v in means_uS.items()}

acc_df = pd.DataFrame({
    'date': means_uS.keys(),
    'mean_uS_cm': means_uS.values(),
    'instrument_accuracy_uS_cm': acc_uS.values()
}).sort_values('date')

print("Accuracy estimada por campaña:")
print(acc_df)

# pares usando diferencias observadas

pairs = [
    ('20220131', '20220816', 1820.515154),
    ('20220131', '20230822', 1471.424532),
    ('20220131', '20250226', 736.498379),
    ('20220131', '20251123', 145.096631),
    ('20220816', '20230822', 349.090622),
    ('20220816', '20250226', 1084.016775),
    ('20220816', '20251123', 1675.418523),
    ('20230822', '20250226', 734.926153),
    ('20230822', '20251123', 1326.327902),
    ('20250226', '20251123', 591.401748),
]

rows = []
for a, b, diff in pairs:
    ua = acc_uS[a]
    ub = acc_uS[b]
# Como se están comparando dos medias, y cada campaña tiene su propia incertidumbre instrumental,
# una forma simple es combinar ambas así: 
    combined_unc = np.sqrt(ua**2 + ub**2)
    robust_vs_instrument = abs(diff) > combined_unc

    rows.append({
        'A': a,
        'B': b,
        'mean_diff_uS_cm': diff,
        'acc_A_uS_cm': ua,
        'acc_B_uS_cm': ub,
        'combined_uncertainty_uS_cm': combined_unc,
        'exceeds_combined_uncertainty': robust_vs_instrument
    })

instrument_check_df = pd.DataFrame(rows)
print("\nComparación contra incertidumbre instrumental:")
print(instrument_check_df)

Accuracy estimada por campaña:
       date    mean_uS_cm  instrument_accuracy_uS_cm
0  20220131  54195.907030                 270.979535
1  20220816  52375.391876                 261.876959
2  20230822  52724.482498                 263.622412
3  20250226  53459.408651                 267.297043
4  20251123  54050.810400                 270.254052

Comparación contra incertidumbre instrumental:
          A         B  mean_diff_uS_cm  acc_A_uS_cm  acc_B_uS_cm  \
0  20220131  20220816      1820.515154   270.979535   261.876959   
1  20220131  20230822      1471.424532   270.979535   263.622412   
2  20220131  20250226       736.498379   270.979535   267.297043   
3  20220131  20251123       145.096631   270.979535   270.254052   
4  20220816  20230822       349.090622   261.876959   263.622412   
5  20220816  20250226      1084.016775   261.876959   267.297043   
6  20220816  20251123      1675.418523   261.876959   270.254052   
7  20230822  20250226       734.926153   263.622412   267.2